<a href="https://colab.research.google.com/github/Aarthiathi16/Android-app-market/blob/main/secure_coding_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ast
import re
from textwrap import indent


code_to_review = r'''
import os
import hashlib
import subprocess
import pickle
import sqlite3

PASSWORD = "admin123"
API_KEY = "12345-SECRET-KEY"

def login(username, password):
    query = "SELECT * FROM users WHERE username='" + username + \
            "' AND password='" + password + "'"

    connection = sqlite3.connect("users.db")
    result = connection.execute(query)

    if password == "admin123":
        return True

    return False

def run_command(user_input):
    subprocess.call("ping " + user_input, shell=True)

def calculate(expression):
    return eval(expression)

def load_data(data):
    return pickle.loads(data)

def create_hash(password):
    return hashlib.md5(password.encode()).hexdigest()

def get_user_data(user_id):
    return os.system("cat /home/user/" + user_id)

DEBUG = True
'''

# ------------------------------------------------------------
# SECURITY RULES
# ------------------------------------------------------------

findings = []

def add_finding(line, category, severity, issue, recommendation):
    findings.append({
        "line": line,
        "category": category,
        "severity": severity,
        "issue": issue,
        "recommendation": recommendation
    })


# ------------------------------------------------------------
# AST-BASED ANALYSIS
# ------------------------------------------------------------

try:
    tree = ast.parse(code_to_review)

    for node in ast.walk(tree):

        # Dangerous eval()
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                function_name = node.func.id

                if function_name == "eval":
                    add_finding(
                        node.lineno,
                        "Code Injection",
                        "HIGH",
                        "Use of eval() can execute attacker-controlled Python code.",
                        "Avoid eval(). Use safe parsing or a restricted set of allowed operations."
                    )

                elif function_name == "exec":
                    add_finding(
                        node.lineno,
                        "Code Injection",
                        "HIGH",
                        "Use of exec() can execute arbitrary Python code.",
                        "Remove exec() and use explicit program logic instead."
                    )

                elif function_name == "input":
                    pass

            # subprocess(..., shell=True)
            if isinstance(node.func, ast.Attribute):
                if node.func.attr in ["call", "run", "Popen"]:

                    for keyword in node.keywords:
                        if (
                            keyword.arg == "shell"
                            and isinstance(keyword.value, ast.Constant)
                            and keyword.value.value is True
                        ):
                            add_finding(
                                node.lineno,
                                "Command Injection",
                                "HIGH",
                                "subprocess is being used with shell=True.",
                                "Avoid shell=True. Pass commands and arguments as a list and validate input."
                            )

                # os.system()
                if node.func.attr == "system":
                    add_finding(
                        node.lineno,
                        "Command Injection",
                        "HIGH",
                        "os.system() executes an operating-system command.",
                        "Use subprocess with shell=False and validated arguments."
                    )

                # pickle.loads()
                if node.func.attr == "loads":
                    if isinstance(node.func.value, ast.Name) and node.func.value.id == "pickle":
                        add_finding(
                            node.lineno,
                            "Unsafe Deserialization",
                            "HIGH",
                            "pickle.loads() can execute malicious code when processing untrusted data.",
                            "Do not deserialize untrusted pickle data. Prefer JSON or another safe serialization format."
                        )

                # hashlib.md5()
                if node.func.attr.lower() == "md5":
                    add_finding(
                        node.lineno,
                        "Weak Cryptography",
                        "MEDIUM",
                        "MD5 is cryptographically broken and should not be used for password hashing.",
                        "Use a password hashing algorithm such as Argon2, bcrypt, or scrypt."
                    )

                # hashlib.sha1()
                if node.func.attr.lower() == "sha1":
                    add_finding(
                        node.lineno,
                        "Weak Cryptography",
                        "MEDIUM",
                        "SHA-1 is not suitable for security-sensitive hashing.",
                        "Use SHA-256 for integrity purposes or Argon2/bcrypt/scrypt for passwords."
                    )


    # --------------------------------------------------------
    # Detect hardcoded credentials
    # --------------------------------------------------------

    for node in ast.walk(tree):

        if isinstance(node, ast.Assign):
            for target in node.targets:

                if isinstance(target, ast.Name):
                    variable = target.id.lower()

                    sensitive_words = [
                        "password",
                        "passwd",
                        "secret",
                        "api_key",
                        "apikey",
                        "token",
                        "private_key"
                    ]

                    if any(word in variable for word in sensitive_words):
                        if isinstance(node.value, ast.Constant):
                            if isinstance(node.value.value, str):
                                add_finding(
                                    node.lineno,
                                    "Hardcoded Secret",
                                    "HIGH",
                                    f"Possible hardcoded secret found in variable '{target.id}'.",
                                    "Store secrets in environment variables or a dedicated secret manager."
                                )

        # Debug mode
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    if target.id.upper() == "DEBUG":
                        if (
                            isinstance(node.value, ast.Constant)
                            and node.value.value is True
                        ):
                            add_finding(
                                node.lineno,
                                "Configuration",
                                "MEDIUM",
                                "DEBUG mode is enabled.",
                                "Disable debug mode in production."
                            )


# ------------------------------------------------------------
# Detect SQL query string concatenation
# ------------------------------------------------------------

    lines = code_to_review.splitlines()

    for i, line in enumerate(lines, 1):
        line_lower = line.lower()

        if (
            ("select " in line_lower or
             "insert " in line_lower or
             "update " in line_lower or
             "delete " in line_lower)
            and ("+" in line)
        ):
            add_finding(
                i,
                "SQL Injection",
                "HIGH",
                "SQL query appears to be constructed using string concatenation.",
                "Use parameterized SQL queries/prepared statements instead of concatenating user input."
            )

except SyntaxError as error:
    print("Syntax error in code being reviewed:")
    print(error)
    raise


# ------------------------------------------------------------
# REMOVE DUPLICATE FINDINGS
# ------------------------------------------------------------

unique_findings = []
seen = set()

for finding in findings:
    key = (
        finding["line"],
        finding["category"],
        finding["issue"]
    )

    if key not in seen:
        seen.add(key)
        unique_findings.append(finding)

findings = sorted(unique_findings, key=lambda x: x["line"])


# ------------------------------------------------------------
# DISPLAY ORIGINAL CODE
# ------------------------------------------------------------

print("=" * 75)
print("🔐 SECURE CODING REVIEW")
print("=" * 75)

print("\n📌 Programming Language: Python")
print("📌 Review Type: Static Security Analysis")

print("\n" + "-" * 75)
print("CODE BEING REVIEWED")
print("-" * 75)

for number, line in enumerate(code_to_review.splitlines(), 1):
    print(f"{number:3}: {line}")


# ------------------------------------------------------------
# DISPLAY FINDINGS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("🚨 SECURITY FINDINGS")
print("=" * 75)

if not findings:
    print("No issues detected by the configured security rules.")
else:

    severity_count = {
        "HIGH": 0,
        "MEDIUM": 0,
        "LOW": 0
    }

    for finding in findings:
        severity_count[finding["severity"]] += 1

        print(f"\n[{finding['severity']}] {finding['category']}")
        print(f"Line           : {finding['line']}")
        print(f"Issue          : {finding['issue']}")
        print(f"Recommendation : {finding['recommendation']}")

    print("\n" + "-" * 75)
    print("SUMMARY")
    print("-" * 75)

    print(f"🔴 HIGH findings   : {severity_count['HIGH']}")
    print(f"🟠 MEDIUM findings : {severity_count['MEDIUM']}")
    print(f"🟢 LOW findings    : {severity_count['LOW']}")
    print(f"📊 Total findings  : {len(findings)}")


# ------------------------------------------------------------
# SECURE CODING RECOMMENDATIONS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("🛡️ SECURE CODING BEST PRACTICES")
print("=" * 75)

recommendations = [
    "Use parameterized SQL queries instead of string concatenation.",
    "Never use eval() or exec() with untrusted input.",
    "Avoid shell=True when executing system commands.",
    "Validate and sanitize all user-controlled input.",
    "Never hardcode passwords, API keys, or tokens in source code.",
    "Use environment variables or a secure secret manager.",
    "Use Argon2, bcrypt, or scrypt for password hashing.",
    "Avoid unsafe deserialization of untrusted data.",
    "Disable DEBUG mode in production.",
    "Keep Python packages and dependencies updated.",
    "Use static analysis tools such as Bandit and Ruff.",
    "Apply least-privilege permissions to files, databases, and services."
]

for number, recommendation in enumerate(recommendations, 1):
    print(f"{number:2}. {recommendation}")


# ------------------------------------------------------------
# FINAL ASSESSMENT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("📋 REVIEW COMPLETED")
print("=" * 75)

if any(f["severity"] == "HIGH" for f in findings):
    print("⚠️ High-severity security issues were detected.")
    print("Recommended action: remediate the HIGH findings before production use.")
elif findings:
    print("⚠️ Security issues were detected. Review the recommendations above.")
else:
    print("✅ No configured security issues were detected.")

print("\nNote: This is a basic educational static analyzer, not a replacement")
print("for a professional security audit or comprehensive SAST tool.")

🔐 SECURE CODING REVIEW

📌 Programming Language: Python
📌 Review Type: Static Security Analysis

---------------------------------------------------------------------------
CODE BEING REVIEWED
---------------------------------------------------------------------------
  1: 
  2: import os
  3: import hashlib
  4: import subprocess
  5: import pickle
  6: import sqlite3
  7: 
  8: PASSWORD = "admin123"
  9: API_KEY = "12345-SECRET-KEY"
 10: 
 11: def login(username, password):
 12:     query = "SELECT * FROM users WHERE username='" + username + \
 13:             "' AND password='" + password + "'"
 14: 
 15:     connection = sqlite3.connect("users.db")
 16:     result = connection.execute(query)
 17: 
 18:     if password == "admin123":
 19:         return True
 20: 
 21:     return False
 22: 
 23: def run_command(user_input):
 24:     subprocess.call("ping " + user_input, shell=True)
 25: 
 26: def calculate(expression):
 27:     return eval(expression)
 28: 
 29: def load_data(data):